**1. Mounting Google Drive in Colab**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**2. Install library**

In [ ]:
!pip install opencv-python pillow scikit-image numpy matplotlib
!pip install rembg onnxruntime
!pip install tensorflow

**3. Import library**

In [ ]:
import os
import math

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf

from PIL import Image
from rembg import remove
from skimage import exposure, data
from skimage.filters import frangi, sato
from skimage.morphology import disk, erosion

**4. Set Dataset and Output Paths**

In [ ]:
# Update this path according to the location of the dataset
base_folder = '/content/drive/MyDrive/In-Ovo_Sexing_Duck_Eggs'

path_folder = os.path.join(
    base_folder,
    'Day_5'
)

preprocessing_folder = os.path.join(
    base_folder,
    'Preprocessing'
)

**5. ROI & Resize: 320 pixel**

In [ ]:
import cv2
import numpy as np
import os

def load_image(path):
    # Read image and convert to RGB
    image_original = cv2.imread(path)
    if image_original is None:
        raise FileNotFoundError(f"Image not found: {path}")
    image_rgb = cv2.cvtColor(image_original, cv2.COLOR_BGR2RGB)
    return image_original, image_rgb

def extract_roi(image_rgb, image_original, target_size=320):
    # Extract ROI from the largest contour and resize to target_size x target_size canvas
    gray = cv2.cvtColor(image_original, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255,cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        raise ValueError("No contour found.")
    largest_contour = max(contours,key=cv2.contourArea)

    # Cropping
    x, y, w, h = cv2.boundingRect(largest_contour)
    roi = image_rgb[y:y+h, x:x+w]

    # Resize ROI while preserving aspect ratio
    scale = min(target_size/w, target_size/h)

    new_w = int(w * scale)
    new_h = int(h * scale)

    roi_resized = cv2.resize(roi,(new_w, new_h),interpolation=cv2.INTER_AREA)

    # Create empty canvas (black)
    canvas = np.zeros((target_size, target_size, 3), dtype=np.uint8)

    # Place ROI at the center of the canvas
    start_x = (target_size - new_w) // 2
    start_y = (target_size - new_h) // 2
    canvas[
        start_y:start_y + new_h,
        start_x:start_x + new_w
    ] = roi_resized

    return canvas

def save_result(canvas, output_folder, filename, target_size=320):
    # Save ROI result to output folder in PNG format
    os.makedirs(output_folder, exist_ok=True)
    base_name = os.path.splitext(filename)[0]
    output_filename = f"{base_name}.png"
    output_path = os.path.join(output_folder, output_filename)
    cv2.imwrite(output_path, cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR))
    return output_filename

def main():
    # Input and output folders
    input_folder = path_folder
    output_folder = os.path.join(preprocessing_folder, 'ROI_Resize')

    # Process all image files in the folder
    for filename in sorted(os.listdir(input_folder)):
        if filename.lower().endswith(('.png')):
            path = os.path.join(input_folder, filename)
            try:
                image_original, image_rgb = load_image(path)
                canvas = extract_roi(image_rgb, image_original, target_size=320)
                save_result(canvas, output_folder, filename, target_size=320)
            except Exception as e:
                print(f"Error processing {filename}: {e}")

if __name__ == "__main__":
    main()

**6. Remove Background**

In [ ]:
import cv2
import numpy as np
import os
from PIL import Image
from rembg import remove

def load_image(path):
    # Load image from path using PIL
    try:
        image = Image.open(path)
        return image
    except Exception as e:
        raise FileNotFoundError(f"Image not found: {path}") from e

def remove_background(input_image):
    # Remove background and convert to RGBA (with transparency)
    output_image_rgba = remove(input_image, alpha_matting=True)
    output_image_rgba = output_image_rgba.convert("RGBA")

    # Create black canvas
    black_background = Image.new("RGB", output_image_rgba.size, (0, 0, 0))
    black_background.paste(output_image_rgba, (0, 0), output_image_rgba)

    return black_background

def save_image(image, output_folder, filename):
    os.makedirs(output_folder, exist_ok=True)
    output_filename = f"{filename}"
    output_path = os.path.join(output_folder, output_filename)
    image.save(output_path)
    return output_path, output_filename

def main():
    input_folder = os.path.join(preprocessing_folder, 'ROI_Resize')
    output_folder = os.path.join(preprocessing_folder, 'Remove_Background')

    all_files = [f for f in os.listdir(input_folder) if f.lower().endswith('.png')]
    for filename in sorted(all_files):
        path = os.path.join(input_folder, filename)
        try:
            input_image = load_image(path)
            result_image = remove_background(input_image)
            output_path, output_filename = save_image(result_image, output_folder, filename)
        except Exception as e:
            print(f"Error processing {filename}: {e}")

if __name__ == "__main__":
    main()

**7. R-component**

In [ ]:
import cv2
import numpy as np
import os
from PIL import Image

def load_image(path):
    # Load image from path and convert to RGB
    image_bgr = cv2.imread(path)
    if image_bgr is None:
        raise FileNotFoundError(f"Image not found: {path}")
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    return image_rgb

def extract_r_component(image_rgb):
    # Extract R channel (Red intensity only) from RGB image
    r_channel = image_rgb[:, :, 0]   # Red channel
    return r_channel

def save_r_component(r_channel, output_folder, filename):
    # Save R channel as grayscale image
    os.makedirs(output_folder, exist_ok=True)
    base_name = os.path.splitext(filename)[0]
    output_filename = f"{base_name}.png"
    output_path = os.path.join(output_folder, output_filename)
    cv2.imwrite(output_path, r_channel)
    return output_path, output_filename

def main():
    input_folder = os.path.join(preprocessing_folder, 'Remove_Background')
    output_folder = os.path.join(preprocessing_folder, 'R_Component')

    all_files = [f for f in os.listdir(input_folder) if f.lower().endswith('.png')]
    # Process all image files
    for filename in sorted(all_files):
        path = os.path.join(input_folder, filename)
        try:
            image_rgb = load_image(path)
            r_channel = extract_r_component(image_rgb)
            save_r_component(r_channel, output_folder, filename)
            print(f"Processed: {filename}")
        except Exception as e:
            print(f"Error processing {filename}: {e}")

if __name__ == "__main__":
    main()

**8. G-component**

In [ ]:
import cv2
import os

def load_image(path):
    # Load image from path and convert to RGB
    image_bgr = cv2.imread(path)
    if image_bgr is None:
        raise FileNotFoundError(f"Image not found: {path}")
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    return image_rgb

def extract_g_component(image_rgb):
    # Extract G channel (Green intensity only) from RGB image
    g_channel = image_rgb[:, :, 1]   # Green channel
    return g_channel

def save_g_component(g_channel, output_folder, filename):
    # Save G channel as grayscale image
    os.makedirs(output_folder, exist_ok=True)
    base_name = os.path.splitext(filename)[0]
    output_filename = f"{base_name}.png"
    output_path = os.path.join(output_folder, output_filename)
    cv2.imwrite(output_path, g_channel)
    return output_path, output_filename

def main():
    input_folder = os.path.join(preprocessing_folder, 'Remove_Background')
    output_folder = os.path.join(preprocessing_folder, 'G_Component')

    all_files = [f for f in os.listdir(input_folder) if f.lower().endswith( '.png')]
    for filename in sorted(all_files):
        path = os.path.join(input_folder, filename)
        try:
            image_rgb = load_image(path)
            g_channel = extract_g_component(image_rgb)
            save_g_component(g_channel, output_folder, filename)
            print(f"Processed: {filename}")
        except Exception as e:
            print(f"Error processing {filename}: {e}")

if __name__ == "__main__":
    main()

**9. R-Binarization**

In [ ]:
import cv2
import os

def load_r_component(path):
    # Read R component image after background removal as grayscale
    image_r = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if image_r is None:
        raise FileNotFoundError(f"Image not found: {path}")
    return image_r

def binarize_r_component(image_r):
    # Preprocessing + Otsu thresholding
    blurred = cv2.GaussianBlur(image_r, (5, 5), 0)
    _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    opened = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel)
    return opened

def save_binary(binary, output_folder, filename):
    os.makedirs(output_folder, exist_ok=True)
    base_name = os.path.splitext(filename)[0]
    output_filename = f"{base_name}.png"
    output_path = os.path.join(output_folder, output_filename)
    cv2.imwrite(output_path, binary)
    return output_path, output_filename

def main():
    input_folder = os.path.join(preprocessing_folder, 'R_Component')
    output_folder = os.path.join(preprocessing_folder, 'Binarization')

    all_files = [f for f in os.listdir(input_folder) if f.lower().endswith('.png')]
    for filename in sorted(all_files):
        path = os.path.join(input_folder, filename)
        try:
            image_r = load_r_component(path)
            binary = binarize_r_component(image_r)
            save_binary(binary, output_folder, filename)
            print(f"Processed: {filename}")
        except Exception as e:
            print(f"Error processing {filename}: {e}")

if __name__ == "__main__":
    main()

**10. Multiplication**

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt

def load_channel(path):
    # Read image channel as grayscale
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Image not found: {path}")
    return img

def multiply_channels(r_img, g_img):
    # Multiply R-binarized × G-component
    multiplied = cv2.multiply(r_img, g_img, scale=1/255.0)
    return multiplied

def save_result(image, output_folder, filename):
    # Save multiplication result
    os.makedirs(output_folder, exist_ok=True)
    base_name = os.path.splitext(filename)[0]
    out_name = f"{base_name}.png"
    out_path = os.path.join(output_folder,out_name)
    cv2.imwrite(out_path, image)
    return out_path, out_name

def main():
    r_folder = os.path.join(preprocessing_folder, 'Binarization')
    g_folder = os.path.join(preprocessing_folder, 'G_Component')
    output_folder = os.path.join(preprocessing_folder, 'Multiplication_Mask')

    files = sorted([
        f for f in os.listdir(r_folder)
        if f.lower().endswith('.png')
    ])

    for filename in files:
        try:
            r_img = load_channel(
                os.path.join(r_folder, filename)
            )
            g_img = load_channel(
                os.path.join(g_folder, filename)
            )

            # Check image dimensions before multiplication
            if r_img.shape != g_img.shape:
                raise ValueError(
                    f"Image size mismatch: {filename}"
                )
            multiplied = multiply_channels(r_img, g_img)
            save_result(multiplied, output_folder,filename)
            print(f"Processed: {filename}")
        except Exception as e:
            print(
                f"Error processing {filename}: {e}"
            )

if __name__ == "__main__":
    main()